In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# دالة IBM Circuit

> **Note:** * دوال Qiskit ميزة تجريبية متاحة فقط لمستخدمي خطط IBM Quantum&reg; Premium Plan وFlex Plan وOn-Prem (عبر IBM Quantum Platform API). وهي في مرحلة إصدار معاينة وقابلة للتغيير.

## نظرة عامة
تأخذ دالة IBM&reg; Circuit كمدخلات [PUBs مجردة](./primitive-input-output)، وتُعيد قيم التوقع المخففة كمخرجات. تتضمن هذه الدالة خط أنابيب آلي ومخصص يُمكّن الباحثين من التركيز على اكتشاف الخوارزميات والتطبيقات.

## الوصف
بعد إرسال PUB الخاص بك، يتم تحويل الدوائر والمراقِبات المجردة تلقائيًا، وتنفيذها على الجهاز الفعلي، ثم معالجتها لاحقًا لإعادة قيم التوقع المخففة. ولتحقيق ذلك، تجمع هذه الدالة بين الأدوات التالية:

- [خدمة Qiskit Transpiler](./qiskit-transpiler-service)، بما في ذلك الاختيار التلقائي لتمريرات الترانسبايل المدعومة بالذكاء الاصطناعي والاستدلالية لترجمة دوائرك المجردة إلى دوائر ISA محسّنة للجهاز
- [تقنيات قمع الأخطاء وتخفيفها اللازمة للحوسبة على نطاق المنفعة](./error-mitigation-and-suppression-techniques)، بما في ذلك تقليب القياس والبوابات، والفصل الديناميكي، وإزالة أخطاء القراءة بالتقليب (TREX)، واستقراء الضوضاء الصفرية (ZNE)، وتضخيم الخطأ الاحتمالي (PEA)
- [Qiskit Runtime Estimator](./get-started-with-primitives)، لتنفيذ PUBs من نوع ISA على الجهاز وإعادة قيم التوقع المخففة

![دالة IBM Circuit](../docs/images/guides/ibm-circuit-function/ibm-circuit-function.svg)

## البدء
اتصل بالخدمة باستخدام [مفتاح API](http://quantum.cloud.ibm.com/) واختر دالة Qiskit كما يلي. (يفترض هذا المقتطف أنك [حفظت حسابك](/guides/functions#install-qiskit-functions-catalog-client) بالفعل في بيئتك المحلية.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("ibm/circuit-function")

## مثال
لتبدأ، جرّب هذا المثال الأساسي:

In [2]:
from qiskit.circuit.random import random_circuit
from qiskit_ibm_runtime import QiskitRuntimeService

# You can skip this step if you have a target backend, e.g.
# backend_name = "ibm_brisbane"
# You'll need to specify the credentials when initializing QiskitRuntimeService, if they were not previously saved.
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit = random_circuit(num_qubits=2, depth=2, seed=42)
observable = "Z" * circuit.num_qubits
pubs = [(circuit, observable)]

job = function.run(
    # Use `backend_name=backend_name` if you didn't initialize a backend object
    backend_name=backend.name,
    pubs=pubs,
)

تحقق من [حالة](/guides/functions#check-job-status) حِمل عمل دالة Qiskit أو استرجع [النتائج](/guides/functions#retrieve-results) كما يلي:

In [3]:
print(job.status())
result = job.result()

QUEUED


The results have the same format as an [Estimator result](/docs/guides/primitive-input-output#estimator-output):

In [4]:
print(f"The result of the submitted job had {len(result)} PUB\n")
print(
    f"The associated PubResult of this job has the following DataBins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB

The associated PubResult of this job has the following DataBins:
 DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(), dtype=float64>))

And this DataBin has attributes: dict_keys(['evs', 'stds', 'ensemble_standard_error'])
The expectation values measured from this PUB are: 
1.02116704805492


النتائج لها نفس صيغة [نتيجة Estimator](/guides/primitive-input-output#estimator-output):

In [5]:
options = {"mitigation_level": 2}

job = function.run(backend_name=backend.name, pubs=pubs, options=options)

#### All available options

In addition to `mitigation_level`, the IBM Circuit function also provides a select number of advanced options that allow you to fine-tune the cost/accuracy trade-off. The following table shows all the available options:

| Option | Sub-option | Sub-sub-option | Description | Choices | Default |
|-|-|-|-|-|-|
| default_precision |  |  | The default precision to use for any PUB or `run()`<br/>call that does not specify one.<br/>Each input PUB can specify its own precision. If the `run()` method is given a precision, then that value is used for all PUBs in the `run()` call that do not specify their own.  | float > 0 | 0.015625 |
| max_execution_time |  |  | Maximum execution time in seconds, which is based<br/>on QPU usage (not wall clock time). QPU usage is the<br/>amount of time that the QPU is dedicated to processing your job. If a job exceeds this time limit, it is forcibly canceled. | Integer number of seconds in the range [1, 10800] |  |
| mitigation_level |  |  | How much error suppression and mitigation to apply. Refer to the [Mitigation level](#mitigation-level) section for more information about the methods used at each level. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | How much optimization to perform on the circuits. [Higher levels](/docs/guides/set-optimization) generate more optimized circuits, at the expense of longer transpilation time. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Whether to enable dynamical decoupling. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) for the explanation of the method.  | True/False | True |
|  | sequence_type |  | Which dynamical decoupling sequence to use.<br/>* `XX`: use the sequence `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: use the sequence `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: use the sequence<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Whether to apply 2-qubit Clifford gate twirling. | True/False | False |
|  | enable_measure |  | Whether to enable twirling of measurements. | True/False | True |
| resilience | measure_mitigation |  | Whether to enable TREX measurement error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) for the explanation of the method.  | True/False | True |
|  | zne_mitigation |  | Whether to turn on Zero Noise Extrapolation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) for the explanation of the method.  | True/False | False |
|  | zne | amplifier | Which technique to use for amplifying noise. One of: <br/> - `gate_folding` (default) uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are chosen randomly.<br/><br/> - `gate_folding_front` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the front of the topologically ordered DAG circuit.<br/><br/> - `gate_folding_back` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the back of the topologically ordered DAG circuit.<br/><br/> - `pea` uses a technique called Probabilistic error amplification (PEA) to amplify noise. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) for the explanation of the method.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Noise factors to use for noise amplification. | list of floats; each float >= 1 | (1, 1.5, 2) for PEA, and (1, 3, 5) otherwise. |
|  |  | extrapolator | Noise factors to evaluate the fit extrapolation models at. This option does not affect execution or model fitting in any way; it only determines the points at which the `extrapolator` objects are evaluated to be returned in the data fields called `evs_extrapolated` and `stds_extrapolated`. | one or more of `exponential`,`linear`, `double_exponential`,`polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Whether to turn on Probabilistic Error Cancellation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) for the explanation of the method.  | True/False | False |
|  | pec | max_overhead | The maximum circuit sampling overhead allowed, or `None` for no maximum. | None/ integer >1 | 100 |

In the following example, setting the mitigation level to 1 initially turns off ZNE mitigation, but setting `zne_mitigation` to `True` overrides the relevant setup from `mitigation_level`.

In [6]:
options = {"mitigation_level": 1, "resilience": {"zne_mitigation": True}}

## المدخلات
راجع الجدول التالي لمعرفة جميع معاملات الإدخال التي تقبلها دالة IBM Circuit. يوضح قسم [الخيارات](#options) التالي مزيدًا من التفاصيل حول `options` المتاحة.

| الاسم | النوع | الوصف | مطلوب | الافتراضي | مثال |
|-----------|----------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|------------------------------------------|
| backend_name   | str                        | اسم الجهاز الخلفي للاستعلام منه. | نعم | غير متاح | `ibm_fez` |
| pubs      | Iterable[EstimatorPubLike] | مجموعة قابلة للتكرار من كائنات تشبه PUB المجردة (الكتلة الموحدة الأولية)، مثل صفوف `(circuit, observables)` أو `(circuit, observables, parameter_values)`. راجع [نظرة عامة على PUBs](/guides/primitive-input-output#overview-of-pubs) للمزيد من المعلومات. يمكن أن تكون الدوائر مجردة (غير ISA). | نعم | غير متاح | (circuit, observables, parameter_values) |
| options   | dict                       | خيارات الإدخال. راجع قسم **الخيارات** لمزيد من التفاصيل. | لا | راجع قسم **الخيارات** للتفاصيل. | `{"optimization_level": 3}` |
| instance  | str                        | الاسم الموردي السحابي للمثيل المراد استخدامه بهذا الشكل. | لا | يُختار واحد عشوائيًا إذا كان حسابك يمتلك صلاحية الوصول إلى عدة مثيلات. | `CRN` |

### الخيارات
#### البنية
على غرار الـ primitives في Qiskit Runtime، يمكن تحديد خيارات دالة IBM Circuit كقاموس متداخل. الخيارات الأكثر استخدامًا، مثل ``optimization_level`` و``mitigation_level``، تكون في المستوى الأول. أما الخيارات الأخرى الأكثر تقدمًا فتُجمَّع في فئات مختلفة، مثل ``resilience``.

#### القيم الافتراضية
إذا لم تحدد قيمة لأحد الخيارات، تُستخدم القيمة الافتراضية التي تحددها الخدمة.

#### مستوى التخفيف
تدعم دالة IBM Circuit أيضًا `mitigation_level`. يحدد مستوى التخفيف مقدار قمع الأخطاء وتخفيفها الذي يُطبَّق على المهمة. المستويات الأعلى تُنتج نتائج أكثر دقة، على حساب أوقات معالجة أطول. تعتمد درجة تخفيض الأخطاء على الطريقة المطبقة. يستخلص مستوى التخفيف الاختيار التفصيلي لأساليب تخفيف الأخطاء وقمعها ليتيح للمستخدمين التفكير في مقايضة التكلفة/الدقة المناسبة لتطبيقهم. يوضح الجدول التالي الأساليب المقابلة لكل مستوى.

> **Note:** على الرغم من تشابه الأسماء، يطبّق `mitigation_level` تقنيات مختلفة عن تلك التي يستخدمها `resilience_level` في Estimator.

على غرار ``resilience_level`` في Qiskit Runtime Estimator، يحدد ``mitigation_level`` مجموعة أساسية من الخيارات المنسّقة. أي خيارات تحددها يدويًا بالإضافة إلى مستوى التخفيف تُطبَّق فوق المجموعة الأساسية من الخيارات التي يحددها مستوى التخفيف. لذلك، من حيث المبدأ، يمكنك تعيين مستوى التخفيف إلى 1 ثم إيقاف تخفيف القياس، وإن كان ذلك غير موصى به.

| **مستوى التخفيف** | **التقنية** |
|:-:|:-:|
| 1 [الافتراضي] | الفصل الديناميكي + تقليب القياس + TREX |
| 2 | المستوى 1 + تقليب البوابات + ZNE عبر طي البوابات |
| 3 | المستوى 1 + تقليب البوابات + ZNE عبر PEA |

يوضح المثال التالي كيفية ضبط مستوى التخفيف:

In [7]:
print(f"The result of the submitted job had {len(result)} PUB")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)
print(f"And the associated metadata is: \n{result[0].metadata}")

The result of the submitted job had 1 PUB
The expectation values measured from this PUB are: 
1.02116704805492
And the associated metadata is: 
{'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32}


#### جميع الخيارات المتاحة
إضافةً إلى `mitigation_level`، توفر دالة IBM Circuit عددًا محددًا من الخيارات المتقدمة التي تتيح لك ضبط مقايضة التكلفة/الدقة بدقة. يوضح الجدول التالي جميع الخيارات المتاحة:

| الخيار | الخيار الفرعي | الخيار الفرعي الثانوي | الوصف | الخيارات المتاحة | الافتراضي |
|-|-|-|-|-|-|
| default_precision |  |  | الدقة الافتراضية المستخدمة لأي PUB أو استدعاء `run()`<br/>لا يحدد دقته الخاصة.<br/>يمكن لكل PUB مدخل تحديد دقته الخاصة. إذا أُعطيت دقة لاستدعاء `run()`، تُستخدم تلك القيمة لجميع PUBs في استدعاء `run()` التي لا تحدد دقتها الخاصة. | float > 0 | 0.015625 |
| max_execution_time |  |  | الحد الأقصى لوقت التنفيذ بالثواني، ويعتمد<br/>على استخدام QPU (وليس وقت الساعة). استخدام QPU هو<br/>الوقت الذي تُكرّس فيه QPU لمعالجة مهمتك. إذا تجاوزت مهمة ما هذا الحد الزمني، يُلغى تنفيذها قسرًا. | عدد صحيح بالثواني في النطاق [1, 10800] |  |
| mitigation_level |  |  | مقدار قمع الأخطاء وتخفيفها المطبّق. راجع قسم [مستوى التخفيف](#mitigation-level) لمزيد من المعلومات حول الأساليب المستخدمة في كل مستوى. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | مقدار التحسين المُجرى على الدوائر. [المستويات الأعلى](/guides/set-optimization) تُنتج دوائر أكثر تحسينًا، على حساب وقت ترانسبايل أطول. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | ما إذا كان يجب تفعيل الفصل الديناميكي. راجع [تقنيات قمع الأخطاء وتخفيفها](/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) لشرح الطريقة. | True/False | True |
|  | sequence_type |  | تسلسل الفصل الديناميكي المراد استخدامه.<br/>* `XX`: استخدام التسلسل `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: استخدام التسلسل `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: استخدام التسلسل<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | ما إذا كان يجب تطبيق تقليب بوابات Clifford ثنائية الكيوبت. | True/False | False |
|  | enable_measure |  | ما إذا كان يجب تفعيل تقليب القياسات. | True/False | True |
| resilience | measure_mitigation |  | ما إذا كان يجب تفعيل طريقة تخفيف أخطاء القياس TREX. راجع [تقنيات قمع الأخطاء وتخفيفها](/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) لشرح الطريقة. | True/False | True |
|  | zne_mitigation |  | ما إذا كان يجب تفعيل طريقة تخفيف الأخطاء باستقراء الضوضاء الصفرية. راجع [تقنيات قمع الأخطاء وتخفيفها](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) لشرح الطريقة. | True/False | False |
|  | zne | amplifier | التقنية المستخدمة لتضخيم الضوضاء. إحدى: <br/> - `gate_folding` (الافتراضي) يستخدم طي البوابات ثنائية الكيوبت لتضخيم الضوضاء. إذا تطلب معامل الضوضاء تضخيم مجموعة فرعية فقط من البوابات، تُختار هذه البوابات عشوائيًا.<br/><br/> - `gate_folding_front` يستخدم طي البوابات ثنائية الكيوبت لتضخيم الضوضاء. إذا تطلب معامل الضوضاء تضخيم مجموعة فرعية فقط من البوابات، تُختار هذه البوابات من مقدمة دائرة DAG المرتبة طوبولوجيًا.<br/><br/> - `gate_folding_back` يستخدم طي البوابات ثنائية الكيوبت لتضخيم الضوضاء. إذا تطلب معامل الضوضاء تضخيم مجموعة فرعية فقط من البوابات، تُختار هذه البوابات من مؤخرة دائرة DAG المرتبة طوبولوجيًا.<br/><br/> - `pea` يستخدم تقنية تُعرف بتضخيم الخطأ الاحتمالي (PEA) لتضخيم الضوضاء. راجع [تقنيات قمع الأخطاء وتخفيفها](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) لشرح الطريقة. | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | معاملات الضوضاء المستخدمة لتضخيم الضوضاء. | قائمة من الأعداد العشرية؛ كل عدد >= 1 | (1, 1.5, 2) لـ PEA، و(1, 3, 5) لغيرها. |
|  |  | extrapolator | معاملات الضوضاء لتقييم نماذج الاستقراء الملائمة عندها. لا يؤثر هذا الخيار على التنفيذ أو ملاءمة النموذج بأي شكل؛ بل يحدد فقط النقاط التي تُقيَّم عندها كائنات `extrapolator` لتُعاد في حقول البيانات المسماة `evs_extrapolated` و`stds_extrapolated`. | واحد أو أكثر من `exponential`، `linear`، `double_exponential`، `polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | ما إذا كان يجب تفعيل طريقة تخفيف الأخطاء بإلغاء الأخطاء الاحتمالي. راجع [تقنيات قمع الأخطاء وتخفيفها](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) لشرح الطريقة. | True/False | False |
|  | pec | max_overhead | الحد الأقصى المسموح به لاستهلاك أخذ عينات الدائرة، أو `None` بدون حد أقصى. | None/ integer >1 | 100 |

في المثال التالي، يُوقف ضبط مستوى التخفيف على 1 تخفيف ZNE في البداية، لكن ضبط `zne_mitigation` على `True` يتجاوز الإعداد ذي الصلة من `mitigation_level`.

In [8]:
job = function.run(
    backend_name="bad_backend_name", pubs=pubs, options=options
)

print(job.result())

QiskitServerlessException: "Traceback (most recent call last):\n  File \"/runner/runner.py\", line 10, in run\n    func = CircuitFunction(**arguments)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/runner/circuit_function/circuit_function.py\", line 87, in __init__\n    self._backend = self._service.backend(\n                    ^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 754, in backend\n    backends = self.backends(name, instance=instance, use_fractional_gates=use_fractional_gates)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 497, in backends\n    raise QiskitBackendNotFoundError(\"No backend matches the criteria.\")\nqiskit.providers.exceptions.QiskitBackendNotFoundError: 'No backend matches the criteria.'\n"

## المخرجات
مخرج دالة Circuit هو [PrimitiveResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult)، الذي يحتوي على حقلين:

- كائن [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) واحد أو أكثر. يمكن فهرستها مباشرةً من `PrimitiveResult`.

- بيانات وصفية على مستوى المهمة.

يحتوي كل `PubResult` على حقل `data` وحقل `metadata`.

- يحتوي حقل `data` على مصفوفة قيم توقع على الأقل (`PubResult.data.evs`) ومصفوفة أخطاء معيارية (`PubResult.data.stds`). ويمكن أن يحتوي أيضًا على بيانات إضافية حسب الخيارات المستخدمة.

- يحتوي حقل `metadata` على بيانات وصفية على مستوى PUB (`PubResult.metadata`).

يصف مقتطف الكود التالي صيغة `PrimitiveResult` (و`PubResult` المرتبط به).